# Outils EN — dataset v5.1 sur le Hub (sonoriser, empaqueter, pousser)

**Runtime : GPU L4.** Secret : `HF_TOKEN` (écriture).

Le livrable est un **dataset prêt à entraîner sur le Hub**, pas un entraînement.
v5.1 ne change qu'une chose par rapport à v5 : la garde stricte des refus
(`docs/v5_report.md`). Le texte transformé est déjà sur le Hub
(`phase_b/train_v5_1.jsonl`). Trois cellules, **une par étape**, chacune reprenable :

1. `voice` — 225 refus en Qwen3-TTS voix Aiden (la voix de l'assistant du corpus) → Hub
2. `pack` — reconstruit tout depuis le Hub, vérifie chaque clip, empaquette (liquid-audio)
3. `push` — les tenseurs packés → `Rcarvalo/tc-en-v5_1-packed` (privé), splits train/val

**Entre `voice` et `pack` : Exécution › Redémarrer la session** — qwen-tts installe ses
propres versions de transformers.


In [ ]:
# Jetons — le plus propre : Colab > icône clé > secrets HF_TOKEN (écriture), GEMINI_API_KEY, WANDB_API_KEY (optionnel).
import os
from getpass import getpass
try:
    from google.colab import userdata
    read = userdata.get
except Exception:
    read = lambda name: getpass(f"{name} : ")
for name, required in (("HF_TOKEN", True), ("GEMINI_API_KEY", False), ("WANDB_API_KEY", False)):
    try:
        value = read(name)
    except Exception:
        value = "" if not required else getpass(f"{name} : ")
    if value:
        os.environ[name] = value
    elif required:
        raise SystemExit(f"{name} manquant")
print("jetons chargés :", [n for n in ("HF_TOKEN", "GEMINI_API_KEY", "WANDB_API_KEY") if os.environ.get(n)])


## 1. Sonoriser les 225 refus (≈ 15 min L4) — reprenable, les clips déjà sur le Hub ne sont pas refaits

In [ ]:
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "tc_en_v51",
    "LFM2_ARGS": "--stage voice",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect,train",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out",
    "TC_EN_TAG": "v5_1",
    "ASSISTANT_VOICE": "Aiden"
})
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")
# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
subprocess.run(["bash", "/content/entry.sh"], check=False)


## 2. Empaqueter (≈ 20 min) — après redémarrage de la session

In [ ]:
# Échoue tôt et avec un décompte si un seul clip manque — jamais une heure plus tard dans le packer.
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "tc_en_v51",
    "LFM2_ARGS": "--stage pack",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect,train",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out",
    "TC_EN_TAG": "v5_1"
})
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")
# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
subprocess.run(["bash", "/content/entry.sh"], check=False)


## 3. Pousser le dataset packé sur le Hub (≈ 5 min)

In [ ]:
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "tc_en_v51",
    "LFM2_ARGS": "--stage push",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect,train",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out",
    "TC_EN_TAG": "v5_1"
})
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")
# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
subprocess.run(["bash", "/content/entry.sh"], check=False)


## (optionnel) Entraîner — décision séparée

≈ 3-5 h L4, 1970 pas, adaptateur poussé sur `Rcarvalo/lfm25-tc-en-v5_1-adapter` tous les
200 pas. Si la VM meurt, relancez avec `--resume-after N` (N = dernier pas poussé).
Ensuite `colab_eval_tc_en.ipynb` pour les portes.


In [ ]:
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "tc_en_v51",
    "LFM2_ARGS": "--stage train",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect,train",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out",
    "TC_EN_TAG": "v5_1"
})
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")
# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
subprocess.run(["bash", "/content/entry.sh"], check=False)
